# **Data Fintering and Querying using pandas**

In [1]:
# Importing required module 
from sqlalchemy import create_engine 
from urllib.parse import quote_plus 
from tabulate import tabulate 
import pandas as pd 
import os 

# Database Creadintial 
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD"))
DB_HOST = "localhost"
DB_PORT = "1433"
DB_NAME = "TestDB"
ODBC_DRIVER = ("ODBC Driver 18 for SQL Server")
TRUST_SERVER_CERTIFICATE = "yes"

# Varify enverment variable
if not DB_USER:
    raise ValueError("DB_USER enverment variable not found")
if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD enverment veriable not found ")

# Database Creadintial for connecting to databse 
connection_string = (
    f"mssql+pyodbc://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    f"?driver={quote_plus(ODBC_DRIVER)}"
    f"&TrustServerCertificate={TRUST_SERVER_CERTIFICATE}"
)

# creating engine for connecting to datbase using connection string 
try : 
    engine = create_engine(connection_string)
    print("Successfully connect to SQL Server databse")

except Exception as error :
    print("Faild to connecting SQL Server Databse")
    raise error 


Successfully connect to SQL Server databse


In [2]:
# querying data for applying filtering logic 
query = """ 
            SELECT 
                * 
            FROM silver.customers ;
        """

In [3]:
# Reating data using pandas dataframe 
try : 
    df = pd.read_sql(query, engine)
    print("Successfully query conplited")

except Exception as error : 
    print("Query faild to datasbe check out the query")
    raise error

Successfully query conplited


#### **Summary of Data quelity check using pandas**

In [4]:
summery  = pd.DataFrame({
    "columns" : df.columns,
    "data_type" : df.dtypes,
    "null_count" : df.isnull().sum(),
    "not_null_count" : df.notnull().sum(),
    "unique_count" : df.nunique(),
    "duplicate_count" : df.apply(lambda col : col.duplicated().sum())
    
})

In [5]:
print(
    tabulate(
        summery,
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌──────────────────────┬──────────────────────┬────────────────┬──────────────┬──────────────────┬────────────────┬───────────────────┐
│                      │ columns              │ data_type      │   null_count │   not_null_count │   unique_count │   duplicate_count │
├──────────────────────┼──────────────────────┼────────────────┼──────────────┼──────────────────┼────────────────┼───────────────────┤
│ customer_id          │ customer_id          │ int64          │            0 │              640 │            640 │                 0 │
├──────────────────────┼──────────────────────┼────────────────┼──────────────┼──────────────────┼────────────────┼───────────────────┤
│ title                │ title                │ object         │          358 │              282 │              5 │               634 │
├──────────────────────┼──────────────────────┼────────────────┼──────────────┼──────────────────┼────────────────┼───────────────────┤
│ first_name           │ first_name           │ 

In [6]:
# breaking dataset using domain logic
customer_identity = ['customer_id', 'title', 'first_name', 'last_name', 'gender', 'is_active']
customer_address = ['customer_id', 'address', 'city', 'state', 'state_abbr', 'zip_code', 'country', 'region']
customer_content = ['customer_id', 'email', 'phone']
customer_business_info = ['customer_id','customer_segment', 'loyalty_points','preferred_channel', 'annual_income_usd', 'company' ]
customer_dates = ['customer_id','date_of_birth', 'account_created_date']

In [7]:
# Creating DataFrame for eatch domain 
try :
    customer_identity_df = df[customer_identity]
    customer_address_df = df[customer_address]
    customer_content_df = df[customer_content]
    customer_business_info_df = df[customer_business_info]
    customer_dates_df = df[customer_dates]

    print("Pandas DataFrame successfully created")

except Exception as erroe : 
    print("Faild to Creating Pandas DataFrame")
    raise error

Pandas DataFrame successfully created


### **Boolean Indexing and filtering data usign pandas** 

In [8]:
# Data Profiling in domain customer_identity DataFrame
print(
    tabulate(
        customer_identity_df.sample(frac=0.01),
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌─────┬───────────────┬─────────┬──────────────┬─────────────┬────────────┬─────────────┐
│     │   customer_id │ title   │ first_name   │ last_name   │ gender     │ is_active   │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼────────────┼─────────────┤
│ 363 │          1364 │ Dr.     │ Eric         │ Sanchez     │ Other      │ False       │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼────────────┼─────────────┤
│ 319 │          1320 │         │ Janet        │ Clark       │ Female     │ False       │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼────────────┼─────────────┤
│ 374 │          1375 │ Ms.     │ Emily        │ Cooper      │ Other      │ False       │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼────────────┼─────────────┤
│ 215 │          1216 │ Prof.   │ Benjamin     │ Sanders     │ Female     │ False       │
├─────┼───────────────┼─────────┼──────────────┼─────────────┼────────────┼─────────────┤
│  99 │   

#### **Applying boolean masking filter**

In [11]:
# returning only those row that contain gender info mail only
print(
    tabulate(
        customer_identity_df[customer_identity_df["gender"] == "Male"].head(),
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌────┬───────────────┬─────────┬──────────────┬─────────────┬──────────┬─────────────┐
│    │   customer_id │ title   │ first_name   │ last_name   │ gender   │ is_active   │
├────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│  2 │          1003 │         │ Justin       │ Alvarez     │ Male     │ True        │
├────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│  4 │          1005 │         │ Debra        │ Wood        │ Male     │ True        │
├────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│  5 │          1006 │         │ Cynthia      │ Moore       │ Male     │ True        │
├────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│  8 │          1009 │ Prof.   │ Thomas       │ Garcia      │ Male     │ False       │
├────┼───────────────┼─────────┼──────────────┼─────────────┼──────────┼─────────────┤
│ 18 │          1019 │         │ Andrew    

#### **Using and(&) Compound conditions**

In [12]:
# filtering those customer that have annual_income_usd > 216294 and annual_income_usd < 218154
print(
    tabulate(
        customer_business_info_df
            [
                (customer_business_info_df['annual_income_usd'] > 216294) &
                (customer_business_info_df['annual_income_usd'] < 218154)
            ],
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌─────┬───────────────┬────────────────────┬──────────────────┬─────────────────────┬─────────────────────┬────────────────────────┐
│     │   customer_id │ customer_segment   │   loyalty_points │ preferred_channel   │   annual_income_usd │ company                │
├─────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼────────────────────────┤
│ 130 │          1131 │ Platinum           │             7270 │ Catalog             │              217005 │ Red Gold Corp          │
├─────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼────────────────────────┤
│ 324 │          1325 │ Bronze             │            14221 │ Catalog             │              217759 │ Global Future LLC      │
├─────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼────────────────────────┤
│ 333 │          1334 │ Platinum           │            13502 │ Websi

#### **Using or(|) Compound conditions**

In [13]:
# filtering those customer that have customer_segment Gold or platinum 
# filtering those customer that have annual_income_usd > 216294 and annual_income_usd < 218154
print(
    tabulate(
        customer_business_info_df
            [
                (customer_business_info_df["customer_segment"] == "Gold") |
                (customer_business_info_df["customer_segment"] == "Platinum")
            ].head(),
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌────┬───────────────┬────────────────────┬──────────────────┬─────────────────────┬─────────────────────┬───────────────┐
│    │   customer_id │ customer_segment   │   loyalty_points │ preferred_channel   │   annual_income_usd │ company       │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼───────────────┤
│  3 │          1004 │ Gold               │             2103 │ Phone Call          │               80521 │ Unknown       │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼───────────────┤
│  4 │          1005 │ Gold               │             6939 │ Phone Call          │              215896 │ Unknown       │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼───────────────┤
│  5 │          1006 │ Gold               │             2536 │ Mobile App          │              142857 │ Unknown       │
├────┼──────────

#### **Using not(~) Compound conditions**

In [14]:
# filtering those customer that have annual_income_usd > 216294 and annual_income_usd < 218154
print(
    tabulate(
        customer_business_info_df
            [
                ~(customer_business_info_df["company"] == "Unknown")
            ].head(),
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌────┬───────────────┬────────────────────┬──────────────────┬─────────────────────┬─────────────────────┬────────────────────────┐
│    │   customer_id │ customer_segment   │   loyalty_points │ preferred_channel   │   annual_income_usd │ company                │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼────────────────────────┤
│  1 │          1002 │ Bronze             │            10485 │ Website             │              236037 │ Blue National Ltd      │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼────────────────────────┤
│  2 │          1003 │ Bronze             │             9064 │ Mobile App          │               54242 │ Summit Metro Inc       │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼────────────────────────┤
│  8 │          1009 │ Platinum           │             3269 │ Website      

#### **Using isin() with Multiple Columns**

In [20]:
print(
    tabulate(
        customer_business_info_df[
            customer_business_info_df["preferred_channel"].isin(
                    ["Mobile App", "Website"]
                    )
            ].head(),
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌────┬───────────────┬────────────────────┬──────────────────┬─────────────────────┬─────────────────────┬───────────────────┐
│    │   customer_id │ customer_segment   │   loyalty_points │ preferred_channel   │   annual_income_usd │ company           │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼───────────────────┤
│  0 │          1001 │ Bronze             │             1519 │ Website             │              216294 │ Unknown           │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼───────────────────┤
│  1 │          1002 │ Bronze             │            10485 │ Website             │              236037 │ Blue National Ltd │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼───────────────────┤
│  2 │          1003 │ Bronze             │             9064 │ Mobile App          │               54242 │ Summ

In [23]:
print(
    tabulate(
        customer_business_info_df[customer_business_info_df["company"].isin(["Gold Blue LLC"])],
        headers='keys',
        tablefmt='simple_grid'
    )
)


┌────┬───────────────┬────────────────────┬──────────────────┬─────────────────────┬─────────────────────┬───────────────┐
│    │   customer_id │ customer_segment   │   loyalty_points │ preferred_channel   │   annual_income_usd │ company       │
├────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼───────────────┤
│  8 │          1009 │ Platinum           │             3269 │ Website             │              198023 │ Gold Blue LLC │
└────┴───────────────┴────────────────────┴──────────────────┴─────────────────────┴─────────────────────┴───────────────┘


#### **using between to filtering data**

In [26]:
print(
    tabulate(
        customer_business_info_df[customer_business_info_df['loyalty_points'].between(3300, 3400)],
        headers='keys',
        tablefmt='simple_grid'
    )
)

┌─────┬───────────────┬────────────────────┬──────────────────┬─────────────────────┬─────────────────────┬──────────────────┐
│     │   customer_id │ customer_segment   │   loyalty_points │ preferred_channel   │   annual_income_usd │ company          │
├─────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼──────────────────┤
│ 301 │          1302 │ Gold               │             3304 │ Mobile App          │              142857 │ Unknown          │
├─────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼──────────────────┤
│ 319 │          1320 │ Bronze             │             3356 │ Website             │              134022 │ Pacific Blue Ltd │
├─────┼───────────────┼────────────────────┼──────────────────┼─────────────────────┼─────────────────────┼──────────────────┤
│ 459 │          1460 │ Silver             │             3326 │ In Store            │               33094 │ Unk

In [11]:
customer_business_info_df.query("loyalty_points > 3300 and loyalty_points < 3400")

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
301,1302,Gold,3304.0,Mobile App,142857.0,Unknown
319,1320,Bronze,3356.0,Website,134022.0,Pacific Blue Ltd
459,1460,Silver,3326.0,In Store,33094.0,Unknown


In [15]:
customer_address_df.query("state_abbr in ['WA', 'CO', 'HO']").head()

,customer_id,address,city,state,state_abbr,zip_code,country,region
0,1001,2548 Washington Blvd,Seattle,Washington,WA,54118,United States,West
1,1002,8752 Commerce Dr Apt 30,Colorado Springs,Colorado,CO,59735,United States,West
17,1018,1626 Washington Blvd,Colorado Springs,Colorado,CO,44687,United States,West
44,1045,3969 Oak Ave,Colorado Springs,Colorado,CO,24404,United States,West
60,1061,3997 School Rd Apt 11,Denver,Colorado,CO,53383,United States,West


In [24]:
customer_business_info_df.query("annual_income_usd.between(200000 , 204000)")

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
214,1215,Silver,4270.0,Mobile App,202347.0,Unknown
357,1358,Bronze,NaN,Website,200483.0,Gold Prime Associates
546,1547,Silver,9899.0,Website,203512.0,Unknown
589,1590,Platinum,56.0,Website,201243.0,Unknown
636,9922,Platinum,56.0,Website,201243.0,Unknown


#### **Reference Python variables with @**

In [27]:
threshold = 204000
customer_business_info_df.query("annual_income_usd >= @threshold").head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1001,Bronze,1519.0,Website,216294.0,Unknown
1,1002,Bronze,10485.0,Website,236037.0,Blue National Ltd
4,1005,Gold,6939.0,Phone Call,215896.0,Unknown
9,1010,Silver,5967.0,Website,213679.0,Unknown
13,1014,Silver,7758.0,Mobile App,226037.0,Prime Prime Associates


In [29]:
company_list = ['Prime Prime Associates', 'Blue National Ltd', 'Gold Prime Associates']
customer_business_info_df.query("company in @company_list")

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
1,1002,Bronze,10485.0,Website,236037.0,Blue National Ltd
13,1014,Silver,7758.0,Mobile App,226037.0,Prime Prime Associates
261,1262,Silver,2663.0,Mobile App,234607.0,Blue National Ltd
357,1358,Bronze,NaN,Website,200483.0,Gold Prime Associates
386,1387,Gold,9550.0,Mobile App,242653.0,Prime Prime Associates


In [31]:
company_list = ['Prime Prime Associates', 'Blue National Ltd', 'Gold Prime Associates']
customer_business_info_df.query("company not in @company_list").head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
0,1001,Bronze,1519.0,Website,216294.0,Unknown
2,1003,Bronze,9064.0,Mobile App,54242.0,Summit Metro Inc
3,1004,Gold,2103.0,Phone Call,80521.0,Unknown
4,1005,Gold,6939.0,Phone Call,215896.0,Unknown
5,1006,Gold,2536.0,Mobile App,142857.0,Unknown


#### **Column names with spaces — use backticks**

In [36]:
customer_business_info_df.query(" `customer_segment` == 'Gold' ").head()

,customer_id,customer_segment,loyalty_points,preferred_channel,annual_income_usd,company
3,1004,Gold,2103.0,Phone Call,80521.0,Unknown
4,1005,Gold,6939.0,Phone Call,215896.0,Unknown
5,1006,Gold,2536.0,Mobile App,142857.0,Unknown
6,1007,Gold,1200.0,In Store,121869.0,Unknown
17,1018,Gold,2573.0,Phone Call,205714.0,Unknown


### **Comparison Methods**

In [ ]:
#  df['col'].eq(5)     # ==    returning woutput in boolean format
#  df['col'].ne(5)     # !=    returning woutput in boolean format
#  df['col'].lt(5)     # <     returning woutput in boolean format
#  df['col'].le(5)     # <=    returning woutput in boolean format
#  df['col'].gt(5)     # >     returning woutput in boolean format
#  df['col'].ge(5)     # >=    returning woutput in boolean format 